In [140]:
import re
import ast

import importlib
import df_preprocessing
import initial_prep

importlib.reload(df_preprocessing)
importlib.reload(initial_prep)

from collections import defaultdict
import copy

import numpy as np
import pandas as pd
from IPython.display import display
import networkx as nx
import seaborn as sns
import matplotlib.pyplot as plt

from df_preprocessing import (
    add_original_to_extended,
    find_all_analogs,
    merge_parts_df1,
    merge_parts_df2,
    prepare_sales,
    normalize,
    normalize_analog_lists,
    article_forms,
    fill_missing_months,
)

from initial_prep import (
    preprocess_repair_parts,
    preprocess_stock_report,
    del_wrong_numbers_df1,
    del_wrong_numbers_df2,
)

In [141]:
df1 = preprocess_repair_parts(r'Запчасти списанные в ремонт.xlsx', 8)
df1 = del_wrong_numbers_df1(df1)

Файл успешно загружен из Excel: Запчасти списанные в ремонт.xlsx


In [ ]:
cols = [
    'Номенклатура.Артикул',
    'Номенклатура.Оригинальный номер',
    'Номенклатура.Оригинальный номер расширенный'
]

for i in cols:
    df1[i] = df1[i].apply(normalize)

df1["Номенклатура.Оригинальный номер расширенный"] = df1.apply(add_original_to_extended, axis=1) 

df1['Аналоги'] = (
    df1['Номенклатура.Оригинальный номер расширенный']
    .fillna('')         
    .str.upper()
    .str.split()
)

graph = defaultdict(set)

for _, row in df1.iterrows():
    # все формы основного артикула (с суффиксом и без)
    part_forms = article_forms(row['Номенклатура.Артикул'])

    # связать формы одного артикула между собой
    for i in range(len(part_forms)):
        for j in range(i + 1, len(part_forms)):
            graph[part_forms[i]].add(part_forms[j])
            graph[part_forms[j]].add(part_forms[i])

    # связать с аналогами
    for analog_raw in row['Аналоги']:
        for af in article_forms(analog_raw):
            for pf in part_forms:
                if pf != af:
                    graph[pf].add(af)
                    graph[af].add(pf)


df1['all_analogs'] = df1['Номенклатура.Артикул'].apply(lambda x: find_all_analogs(x, graph))
df1 = df1.drop(columns='Аналоги')

group_mapping = {}
group_counter = 1

for group_tuple in df1['all_analogs'].unique():
    group_mapping[group_tuple] = group_counter
    group_counter += 1

df1['Номер группы'] = df1['all_analogs'].apply(lambda x: group_mapping.get(x))
df_quarter = merge_parts_df1(df1)

IndexError: tuple index out of range

In [143]:
df2 = preprocess_stock_report(r'Остатки и обороты.xlsx', skiprows=8)

Файл успешно загружен из Excel: Остатки и обороты.xlsx


In [144]:
df2 = del_wrong_numbers_df2(df2)

In [145]:
df2["Артикул"]  = df2["Артикул"].apply(normalize)
df2["Оригинальный номер"] = df2["Оригинальный номер"].apply(normalize)

def lookup(art_norm, orig_norm):
    for val in [art_norm, orig_norm]:
        if not val:
            continue
        for form in article_forms(val):
            if form in article_to_group:
                return article_to_group[form], article_to_analogs[form]
    return None, None

article_to_group   = {}
article_to_analogs = {}

for _, row in df1.drop_duplicates('Номенклатура.Артикул').iterrows():
    group = row['Номер группы']
    raw = row['all_analogs']
    try:
        analogs = raw if isinstance(raw, tuple) else ast.literal_eval(raw)
    except:
        analogs = ()
    for art in analogs:
        article_to_group[art]   = group
        article_to_analogs[art] = analogs
    main = row['Номенклатура.Артикул']
    if pd.notna(main) and main:
        article_to_group[main]   = group
        article_to_analogs[main] = analogs


groups, analogs_list = zip(
    *df2.apply(lambda r: lookup(r["Артикул"], r["Оригинальный номер"]), axis=1)
)

df2["Номер группы"] = groups
df2["Список аналогов"] = analogs_list

def enrich_analogs(row):
    art = row["Артикул"]
    analogs = row["Список аналогов"]
    if pd.isnull(art) or not isinstance(analogs, tuple):
        return analogs
    if art not in analogs and art not in article_to_group:
        return tuple(sorted(analogs + (art,)))
    return analogs

df2["Список аналогов"] = df2.apply(enrich_analogs, axis=1)
df2 = normalize_analog_lists(df2)

In [146]:
graph_new = defaultdict(set)

for idx in df2[df2["Номер группы"].isna()].index:
    art = normalize(df2.at[idx, "Артикул"])
    orig = normalize(df2.at[idx, "Оригинальный номер"])

    if art is None:
        if orig is not None:
            art = orig
        else:
            continue

    # проверяем все формы, не только точную
    all_forms = article_forms(art) + (article_forms(orig) if orig else [])
    if any(f in article_to_group for f in all_forms):
        continue

    for af in article_forms(art):
        for bf in article_forms(art):
            if af != bf:
                graph_new[af].add(bf)
                graph_new[bf].add(af)

    if orig is not None:
        for af in article_forms(art):
            for bf in article_forms(orig):
                if af != bf:
                    graph_new[af].add(bf)
                    graph_new[bf].add(af)


art_by_group = (
    df2[df2["Номер группы"].notna()]
    .assign(Артикул_norm=lambda d: d["Артикул"].apply(normalize))
    .dropna(subset=["Артикул_norm"])
    .groupby(df2["Номер группы"].dropna().astype(str))["Артикул_norm"]
    .apply(set)
    .to_dict()
)

df1_group_idx = defaultdict(list)
for i, g in df1["Номер группы"].astype(str).items():
    df1_group_idx[g].append(i)

df2_group_idx = defaultdict(list)
for i, g in df2["Номер группы"].dropna().astype(str).items():
    df2_group_idx[g].append(i)

for grp, arts in art_by_group.items():
    idxs_df1 = df1_group_idx.get(grp, [])
    if not idxs_df1:
        continue
    existing = df1.at[idxs_df1[0], "all_analogs"]
    if not isinstance(existing, tuple):
        continue
    new_arts = arts - set(existing)
    if not new_arts:
        continue
    new_analogs = tuple(sorted(existing + tuple(new_arts)))
    for i in idxs_df1:
        df1.at[i, "all_analogs"] = new_analogs
    for i in df2_group_idx.get(grp, []):
        df2.at[i, "Список аналогов"] = new_analogs


for idx in df2[df2["Номер группы"].isna()].index:
    art = normalize(df2.at[idx, "Артикул"])
    orig = normalize(df2.at[idx, "Оригинальный номер"])

    if art is None:
        art = orig

    all_forms = article_forms(art) if art else []
    if any(f in article_to_group for f in all_forms):
        continue

    if art is not None:
        df2.at[idx, "Список аналогов"] = find_all_analogs(art, graph_new)
    else:
        df2.at[idx, "Список аналогов"] = tuple()


unique_new_analogs = df2.loc[df2["Номер группы"].isna(), "Список аналогов"].drop_duplicates()
new_group = df1["Номер группы"].max() + 1
group_mapping = {grp: new_group + i for i, grp in enumerate(unique_new_analogs)}

mask = df2["Номер группы"].isna()
df2.loc[mask, "Номер группы"] = df2.loc[mask, "Список аналогов"].apply(
    lambda x: group_mapping.get(x)
)

In [147]:
agg = merge_parts_df2(df2)
agg = prepare_sales(agg, df_quarter)

In [148]:
final_df = pd.merge(
    df_quarter, 
    agg, 
    on=['Год','Месяц','Номер группы'], 
    how='outer', 
    suffixes=('_df','_agg'),
)

cols_to_combine = [c for c in df_quarter.columns if c in agg.columns and c not in ['Год','Месяц','Номер группы']]

for col in cols_to_combine:
    final_df[col] = final_df[f'{col}_df'].combine_first(final_df[f'{col}_agg'])

cols_to_drop = [c for c in final_df.columns if c.endswith('_df') or c.endswith('_agg')]

final_df.drop(columns=cols_to_drop, inplace=True)

final_df['Продажа'] = final_df['Продажа'].fillna(0)
final_df['Ремонт'] = final_df['Ремонт'].fillna(0)

cols = [c for c in final_df.columns if c not in ['Продажа', 'Ремонт', 'Приход', 'Конечный остаток']]

final_df = final_df[cols + ['Приход', 'Конечный остаток', 'Продажа', 'Ремонт']]

final_df = normalize_analog_lists(
    final_df,
    col_group="Номер группы",
    col_analogs="Список аналогов"
)

name_map = (
    final_df.groupby('Номер группы')['Номенклатура']
    .apply(lambda s: min(s.dropna().astype(str), key=len, default=None))
)
final_df['Номенклатура'] = final_df['Номер группы'].map(name_map)

In [ ]:
final_df = fill_missing_months(final_df)
final_df.to_excel('final.xlsx', index=False)